# Case Study 1: Shelf-Life Prediction with a Keras MLP
Use `pandas` for food physicochemical data, `numpy` for arrays and standardisation, and `tensorflow.keras` for model development.

<a href="https://colab.research.google.com/github/khairuladib94/dl-food-research/blob/main/session-6/notebooks/01_tabular_mlp_shelf_life_tensorflow_keras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

> **Colab workflow:** Save a copy in Drive, then choose Runtime > Run all. The required teaching dataset downloads automatically; Google Drive is not mounted.



## Learning route

This regression example follows the complete modelling workflow. The MLP receives one row of physicochemical and storage measurements at a time and predicts shelf life in days.

![From food data to an evaluated model](../assets/session6-model-workflow.png)

The visual is a useful checklist: understand the data, create a protected test set, preprocess using training statistics, train, and only then evaluate.


## 1. Prepare the notebook environment

We begin with the libraries used throughout the case study. Setting TensorFlow's log level before importing TensorFlow keeps the workshop output focused on the results.


In [ ]:
from pathlib import Path
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


### Make the data portable

The next cell first looks for `tabular_shelf_life.csv` beside the notebook. If the file is absent—as it will be in a fresh Colab session—it downloads the public workshop copy. This keeps the notebook runnable without mounting Google Drive.


In [ ]:
# Portable workshop data setup: local Jupyter first, Colab fallback.
DATA_FILE = 'tabular_shelf_life.csv'
LOCAL_DATA_DIR = Path('../data')
if (LOCAL_DATA_DIR / DATA_FILE).exists():
    DATA_DIR = LOCAL_DATA_DIR
    IN_COLAB = False
else:
    from urllib.request import urlretrieve
    try:
        import google.colab  # type: ignore  # noqa: F401
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False
    DATA_DIR = Path('/content/dl-food-research-data') if IN_COLAB else Path.cwd() / '.workshop-data'
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    target = DATA_DIR / DATA_FILE
    if not target.exists():
        url = 'https://raw.githubusercontent.com/khairuladib94/dl-food-research/main' + '/session-6/data/' + DATA_FILE
        print(f'Downloading {DATA_FILE} ...')
        urlretrieve(url, target)
    print(f'Dataset ready: {target}')


### Make the experiment repeatable

Fixed random seeds make the teaching run easier to reproduce. The helper functions also keep splitting, standardisation, and training-history summaries consistent across the five examples.

> **Important:** standardisation statistics are calculated from the training set only. Using the test set here would leak information into the model.


In [ ]:
SEED = 7
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)
print('TensorFlow:', tf.__version__)
print('Keras:', keras.__version__)

def make_split(n, test_fraction=0.2):
    idx = np.arange(n)
    np.random.shuffle(idx)
    cut = int(n * (1 - test_fraction))
    return idx[:cut], idx[cut:]

def standardize(train, test, axis=0):
    mean = train.mean(axis=axis, keepdims=True)
    std = train.std(axis=axis, keepdims=True) + 1e-8
    return (train - mean) / std, (test - mean) / std, mean, std

def history_table(history):
    hist = pd.DataFrame(history.history)
    return hist.tail(5).round(4)


### Set up reusable plots

These helpers turn Keras histories and classification results into readable diagnostics. They do not affect training; they only help us inspect what the model learned.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (8, 4.8),
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.size': 10
})

def plot_history(history, metrics=None, title='Training history'):
    hist = pd.DataFrame(history.history)
    metrics = metrics or [c for c in hist.columns if not c.startswith('val_')]
    fig, axes = plt.subplots(1, len(metrics), figsize=(6 * len(metrics), 4))
    if len(metrics) == 1:
        axes = [axes]
    for ax, metric in zip(axes, metrics):
        ax.plot(hist.index + 1, hist[metric], label=f'train {metric}', linewidth=2)
        val_metric = f'val_{metric}'
        if val_metric in hist:
            ax.plot(hist.index + 1, hist[val_metric], label=f'validation {metric}', linewidth=2)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(metric)
        ax.set_title(metric)
        ax.legend()
    fig.suptitle(title, y=1.03, fontweight='bold')
    plt.tight_layout()
    plt.show()

def plot_binary_confusion(y_true, y_pred, labels, title='Confusion matrix'):
    cm = confusion_matrix(y_true.astype(int), y_pred.astype(int))
    disp = ConfusionMatrixDisplay(cm, display_labels=labels)
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    return cm


## 2. Load and inspect the table

Before modelling, verify the number of rows, column types, plausible ranges, and missing or surprising values.


In [ ]:
df = pd.read_csv(DATA_DIR / 'tabular_shelf_life.csv')
print(df.shape)
display(df.head())
display(df.select_dtypes(include='number').describe().round(2))


### Explore distributions and relationships

Plots reveal class balance, target spread, and possible relationships. The correlation map is descriptive—not proof of causality.


In [ ]:
numeric_cols = ['moisture', 'pH', 'brix', 'protein', 'storage_temp', 'water_activity', 'packaging_o2', 'shelf_life_days']

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
df['food_type'].value_counts().plot(kind='bar', ax=axes[0], color='#0B2A55')
axes[0].set_title('Samples by food category')
axes[0].set_xlabel('')
axes[0].set_ylabel('Count')

axes[1].hist(df['shelf_life_days'], bins=18, color='#3E7CB1', edgecolor='white')
axes[1].set_title('Shelf-life distribution')
axes[1].set_xlabel('Shelf life (days)')
axes[1].set_ylabel('Samples')

scatter = axes[2].scatter(
    df['moisture'], df['shelf_life_days'],
    c=df['storage_temp'], cmap='viridis', s=45, alpha=0.85
)
axes[2].set_title('Moisture vs shelf life')
axes[2].set_xlabel('Moisture (%)')
axes[2].set_ylabel('Shelf life (days)')
fig.colorbar(scatter, ax=axes[2], label='Storage temp (C)')
plt.tight_layout()
plt.show()


### Examine correlations

The heat map summarizes pairwise linear relationships among numerical variables. Strong correlation can suggest useful predictors or redundant measurements.


In [ ]:
corr = df[numeric_cols].corr()
fig, ax = plt.subplots(figsize=(7.2, 5.6))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr.columns)), corr.columns)
ax.set_title('Correlation map for measured variables')
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.82)
plt.tight_layout()
plt.show()


## 3. Encode the predictors and define the target

One-hot encoding converts `food_type` into numerical indicator columns. We keep shelf life separate as the regression target.


In [ ]:
df_model = pd.get_dummies(df, columns=['food_type'], drop_first=False)
target = 'shelf_life_days'
X = df_model.drop(columns=[target]).to_numpy(dtype='float32')
y = df_model[[target]].to_numpy(dtype='float32')


### Split before standardising

The held-out test rows simulate unseen samples. Both predictors and target are standardised using training-set statistics only.


In [ ]:
train_idx, test_idx = make_split(len(df_model))
X_train_raw, X_test_raw = X[train_idx], X[test_idx]
y_train_raw, y_test_raw = y[train_idx], y[test_idx]

X_train, X_test, X_mean, X_std = standardize(X_train_raw, X_test_raw, axis=0)
y_train, y_test, y_mean, y_std = standardize(y_train_raw, y_test_raw, axis=0)
print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)


## 4. Build the MLP

Dense layers combine all tabular features. The final layer has no activation because shelf life is a continuous value.


In [ ]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.10),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])


### Choose the learning objective

Mean squared error trains the network, while mean absolute error gives an additional, more interpretable view of typical error.


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss='mse',
    metrics=[keras.metrics.MeanAbsoluteError(name='mae')]
)
model.summary()


## 5. Train the model

During each epoch, Keras updates weights on the training subset and reports performance on a validation subset.


In [ ]:
history = model.fit(
    X_train, y_train,
    validation_split=0.20,
    epochs=45,
    batch_size=32,
    verbose=0
)
history_table(history)


### Read the learning curves

Training and validation curves should improve together. A widening gap can indicate overfitting.


In [ ]:
plot_history(history, metrics=['loss', 'mae'], title='MLP shelf-life model learning curves')


## 6. Return predictions to days

Because the target was standardised, predictions must be transformed back before MAE and RMSE are reported in days.


In [ ]:
pred_scaled = model.predict(X_test, verbose=0)
pred_days = pred_scaled * y_std + y_mean
true_days = y_test_raw

mae = np.mean(np.abs(pred_days - true_days))
rmse = np.sqrt(np.mean((pred_days - true_days) ** 2))
print(f'Test MAE: {mae:.2f} days')
print(f'Test RMSE: {rmse:.2f} days')


### Inspect individual predictions

A small table makes model errors concrete and helps identify samples that deserve investigation.


In [ ]:
pd.DataFrame({
    'true_days': true_days[:12, 0],
    'predicted_days': pred_days[:12, 0],
    'absolute_error': np.abs(pred_days[:12, 0] - true_days[:12, 0])
}).round(2)


### Compare measured and predicted shelf life

Points near the diagonal are accurate. The error histogram shows whether mistakes are centered near zero or systematically biased.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
axes[0].scatter(true_days.ravel(), pred_days.ravel(), color='#0B2A55', alpha=0.8)
lo = min(true_days.min(), pred_days.min())
hi = max(true_days.max(), pred_days.max())
axes[0].plot([lo, hi], [lo, hi], '--', color='#D1495B', linewidth=2)
axes[0].set_title('Predicted vs measured shelf life')
axes[0].set_xlabel('Measured shelf life (days)')
axes[0].set_ylabel('Predicted shelf life (days)')

errors = (pred_days - true_days).ravel()
axes[1].hist(errors, bins=16, color='#3E7CB1', edgecolor='white')
axes[1].axvline(0, color='#D1495B', linestyle='--', linewidth=2)
axes[1].set_title('Prediction error distribution')
axes[1].set_xlabel('Prediction error (days)')
axes[1].set_ylabel('Samples')
plt.tight_layout()
plt.show()


## How to read the evidence

![Three complementary views of model quality](../assets/session6-evaluation-views.png)

Learning curves describe optimization, predictions show sample-level behaviour, and error summaries expose the kinds of mistakes being made. A publishable conclusion needs all three, plus external validation that matches the intended use.


## Teaching notes
- Keep a tree-based or linear baseline for comparison in publications.
- Split by batch, supplier, production date, or season when possible.
- Report MAE/RMSE in original units, not only scaled loss.